## Clinical Variables — Demographic Table

Variables: `hy`, `educ`, `gds`, `duration` (de novo PD only), `agediag_hive` (de novo PD only),
`moca`, `updrs3_score`, `Field Strength`, `age`, `SEX`.

Three groups: HC (2), de novo PD (1), prodromal PD (4).

**Format:**
- Continuous variables → Median (IQR)
- Categorical variables → n (%)
- Global test p-value per variable
- Bonferroni-corrected post-hoc comparisons where significant

**NOTE:** `hy` = Reclassified Hoehn & Yahr stage: 0 = stage 0, 1 = stage 1, 2 = stage 2, 3 = stages 3–5.

**Table format (thesis-ready):**
- Continuous: Median (IQR)
- Categorical: n (%)
- Global p-value column
- Post-hoc column (Bonferroni-corrected, group names)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
%pip install openpyxl -q  # required for .xlsx export

from pathlib import Path
import numpy as np
import pandas as pd
from itertools import combinations

# Statistical tests
from scipy.stats import kruskal, chi2_contingency, mannwhitneyu
# kruskal       : non-parametric omnibus test across ≥2 groups (equivalent of one-way ANOVA)
# mannwhitneyu  : non-parametric pairwise test (post-hoc follow-up to Kruskal-Wallis)
# chi2_contingency : chi-squared test of independence for categorical variables

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH    = Path("../../data/df1.csv")
RESULTS_PATH = Path("../../results")

# ── Load data ─────────────────────────────────────────────────────────────────
# low_memory=False suppresses mixed-type DtypeWarning on large files
df1 = pd.read_csv(DATA_PATH, low_memory=False)
raw_df1 = df1.copy()

In [ ]:
# ── Settings ──────────────────────────────────────────────────────────────────
group_col   = "CONCOHORT"       # column defining group membership
group_order = [4, 1, 2]         # display order: Prodromal PD, De Novo PD, Control

# Human-readable group labels used in post-hoc and column headers
GROUP_LABELS = {
    1: "de novo PD",
    2: "Control",
    4: "prodromal PD",
}

# Full variable list in desired display order, with type tag
# Types: "continuous" | "pd_only" | "categorical"
VAR_ORDER = [
    ("age",            "continuous"),
    ("agediag_hive",   "pd_only"),      # De Novo PD only
    ("duration",       "pd_only"),      # De Novo PD only
    ("SEX",            "categorical"),
    ("EDUCYRS",        "continuous"),
    ("updrs3_score",   "continuous"),
    ("moca",           "continuous"),   # adjusted for education in PPMI
    ("gds",            "continuous"),
    ("hy",             "categorical"),  # ordinal; counts per stage
    ("Field Strength", "categorical"),
]

# Group that provides pd_only variables
denovo_only_for_groups = [1]

# Fixed category orders for ordinal/binary categoricals
hy_order             = [0, 1, 2, 3]
field_strength_order = [1.5, 3.0]
sex_order            = [0, 1]          # 0 = Female, 1 = Male

category_orders = {
    "SEX":            sex_order,
    "hy":             hy_order,
    "Field Strength": field_strength_order,
}

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

def format_p(p: float) -> str:
    """
    Format a p-value for publication.
    Values below 0.001 are reported as '< 0.001' per APA/journal convention.
    """
    if pd.isna(p):
        return "NA"
    if p < 0.001:
        return "p < 0.001"
    return f"p = {p:.3f}"


def median_iqr(series: pd.Series) -> str:
    """
    Return 'Median (IQR)' string for a continuous variable.
    IQR = Q3 - Q1.
    """
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return "NA"
    med = s.median()
    iqr = s.quantile(0.75) - s.quantile(0.25)
    return f"{med:.2f} ({iqr:.2f})"


def counts_pct(series: pd.Series, category_order: list = None) -> str:
    """
    Return 'n (%)' counts for each category level, separated by ' / '.
    Percentages are calculated over non-missing values within the group.
    """
    s = series.dropna()
    n_total = len(s)
    if n_total == 0:
        return "NA"

    if category_order is None:
        counts = s.value_counts().sort_index()
        return " / ".join(f"{int(c)} ({100 * c / n_total:.1f}%)" for c in counts.values)

    parts = []
    for cat in category_order:
        c = (s == cat).sum()
        parts.append(f"{int(c)} ({100 * c / n_total:.1f}%)")
    return " / ".join(parts)


def kruskal_with_posthoc(
    data: pd.DataFrame,
    var: str,
    group_col: str,
    groups: list,
    alpha: float = 0.05,
) -> tuple[str, str]:
    """
    Kruskal-Wallis omnibus test + Bonferroni-corrected Mann-Whitney U post-hoc.

    Kruskal-Wallis H-test:
      - Non-parametric alternative to one-way ANOVA.
      - Tests whether the medians of ≥2 independent groups differ.
      - Does not assume normality; uses rank sums.

    Post-hoc — Mann-Whitney U (Wilcoxon rank-sum):
      - Pairwise non-parametric test comparing two groups.
      - Bonferroni correction: threshold = alpha / n_pairs
        (with 3 groups → 3 pairs → threshold = 0.05 / 3 ≈ 0.017).
      - Only run when the omnibus Kruskal-Wallis is significant.
    """
    arrays, valid_groups = [], []
    for g in groups:
        vals = pd.to_numeric(data.loc[data[group_col] == g, var], errors="coerce").dropna()
        if len(vals) > 0:
            arrays.append(vals)
            valid_groups.append(g)

    if len(arrays) < 2:
        return "NA", ""

    stat, p = kruskal(*arrays)
    stat_str = f"H = {stat:.2f}, {format_p(p)}"

    # Only run post-hoc if omnibus is significant
    if p >= alpha:
        return stat_str, ""

    pairs = list(combinations(valid_groups, 2))
    bonferroni_threshold = alpha / len(pairs)  # Bonferroni correction
    posthoc_labels = []

    for g1, g2 in pairs:
        x1 = pd.to_numeric(data.loc[data[group_col] == g1, var], errors="coerce").dropna()
        x2 = pd.to_numeric(data.loc[data[group_col] == g2, var], errors="coerce").dropna()
        if len(x1) > 0 and len(x2) > 0:
            _, p_pair = mannwhitneyu(x1, x2, alternative="two-sided")
            if p_pair < bonferroni_threshold:
                lbl1 = GROUP_LABELS.get(g1, str(g1))
                lbl2 = GROUP_LABELS.get(g2, str(g2))
                posthoc_labels.append(f"{lbl1} vs {lbl2} ({format_p(p_pair)})")

    return stat_str, "; ".join(posthoc_labels)


def chi2_with_posthoc(
    data: pd.DataFrame,
    var: str,
    group_col: str,
    groups: list,
    alpha: float = 0.05,
) -> tuple[str, str]:
    """
    Chi-squared test of independence + Bonferroni-corrected pairwise chi-squared post-hoc.

    Chi-squared test:
      - Tests whether the distribution of a categorical variable differs across groups.
      - Uses a contingency table (groups × categories).

    Post-hoc — pairwise chi-squared:
      - Each pair of groups is tested in a 2×k contingency table.
      - Bonferroni correction applied: threshold = alpha / n_pairs.
      - Only run when the omnibus chi-squared is significant.
    """
    sub = data[data[group_col].isin(groups)][[group_col, var]].dropna()
    if sub.empty:
        return "NA", ""

    table = pd.crosstab(sub[group_col], sub[var])
    if table.shape[0] < 2 or table.shape[1] < 2:
        return "NA", ""

    chi2, p, _, _ = chi2_contingency(table)
    stat_str = f"χ² = {chi2:.2f}, {format_p(p)}"

    # Only run post-hoc if omnibus is significant
    if p >= alpha:
        return stat_str, ""

    pairs = list(combinations(groups, 2))
    bonferroni_threshold = alpha / len(pairs)
    posthoc_labels = []

    for g1, g2 in pairs:
        pair_sub = sub[sub[group_col].isin([g1, g2])]
        pair_tab = pd.crosstab(pair_sub[group_col], pair_sub[var])
        if pair_tab.shape[0] == 2 and pair_tab.shape[1] >= 2:
            try:
                _, p_pair, _, _ = chi2_contingency(pair_tab)
                if p_pair < bonferroni_threshold:
                    lbl1 = GROUP_LABELS.get(g1, str(g1))
                    lbl2 = GROUP_LABELS.get(g2, str(g2))
                    posthoc_labels.append(f"{lbl1} vs {lbl2} ({format_p(p_pair)})")
            except Exception:
                pass

    return stat_str, "; ".join(posthoc_labels)

In [ ]:
# ── Build Table ───────────────────────────────────────────────────────────────
rows = []

for var, var_type in VAR_ORDER:

    row = {"Variable": var}

    if var_type == "continuous":
        # Median (IQR) for all groups, Kruskal-Wallis omnibus + Mann-Whitney post-hoc
        for g in group_order:
            row[g] = median_iqr(df1.loc[df1[group_col] == g, var])
        stat_str, posthoc = kruskal_with_posthoc(df1, var, group_col, group_order)

    elif var_type == "pd_only":
        # Only De Novo PD has data — no cross-group test possible
        for g in group_order:
            if g in denovo_only_for_groups:
                row[g] = median_iqr(df1.loc[df1[group_col] == g, var])
            else:
                row[g] = "—"   # em-dash: not applicable for this group
        stat_str, posthoc = "—", ""

    elif var_type == "categorical":
        # n (%) per category level, Chi-squared omnibus + pairwise chi-squared post-hoc
        cat_order = category_orders.get(var, None)
        for g in group_order:
            row[g] = counts_pct(df1.loc[df1[group_col] == g, var], cat_order)
        stat_str, posthoc = chi2_with_posthoc(df1, var, group_col, group_order)

    row["Statistics"] = stat_str
    row["Post-hoc (Bonferroni)"] = posthoc
    rows.append(row)

# ── Assemble DataFrame ────────────────────────────────────────────────────────
table1 = pd.DataFrame(rows)
table1 = table1[["Variable"] + group_order + ["Statistics", "Post-hoc (Bonferroni)"]]

# Column headers with group N
col_labels = {
    4: f"prodromal PD (N={df1.loc[df1[group_col] == 4].shape[0]})",
    1: f"de novo PD (N={df1.loc[df1[group_col] == 1].shape[0]})",
    2: f"Control (N={df1.loc[df1[group_col] == 2].shape[0]})",
}
table1 = table1.rename(columns=col_labels)

# Readable variable names
pretty_names = {
    "age":           "Age, years",
    "agediag_hive":  "Age at diagnosis, years *",
    "duration":      "Disease duration, years *",
    "SEX":           "Sex, Female / Male, n (%)",
    "EDUCYRS":       "Education, years",
    "updrs3_score":  "UPDRS-III score",
    "moca":          "MoCA score",
    "gds":           "GDS score",
    "hy":            "Hoehn & Yahr (0 / 1 / 2 / 3), n (%)",
    "Field Strength": "Field strength (1.5T / 3T), n (%)",
}
table1["Variable"] = table1["Variable"].replace(pretty_names)

display(table1)

print("\n* de novo PD only — no cross-group statistical test applicable.")
print("Continuous variables reported as Median (IQR).")
print("Global test: Kruskal-Wallis H-test (continuous) / Chi-squared (categorical).")
print("Post-hoc: Mann-Whitney U (continuous) / pairwise chi-squared (categorical), Bonferroni-corrected.")

In [ ]:
# ── Save outputs ──────────────────────────────────────────────────────────────
# Paths are relative to this notebook: notebooks/02_clinical/ → ../../results/
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

csv_out   = RESULTS_PATH / "clinical_summary_table.csv"
excel_out = RESULTS_PATH / "clinical_summary_table.xlsx"

table1.to_csv(csv_out, index=False)
table1.to_excel(excel_out, index=False)

print(f"Saved: {csv_out}")
print(f"Saved: {excel_out}")

In [ ]:
# ── SEX coding refresher ──────────────────────────────────────────────────────
# 0 = Female, 1 = Male
print("SEX unique values:", sorted(df1["SEX"].dropna().unique()))
print(df1["SEX"].value_counts(dropna=False).sort_index())

In [ ]:
# ── Sanity check: duration and agediag_hive should only exist in PD group ─────
print(df1.groupby("CONCOHORT")[["duration", "agediag_hive"]].count())

## Notes

**Statistical tests used:**
- **Kruskal-Wallis H-test** (continuous variables): Non-parametric omnibus test that compares the rank-based distributions across three groups. Used because normality cannot be assumed for all variables. Analogous to a one-way ANOVA on ranks.
- **Mann-Whitney U test** (post-hoc, continuous): Pairwise non-parametric test comparing the rank distributions of two groups. Applied as follow-up to a significant Kruskal-Wallis result.
- **Chi-squared test of independence** (categorical variables): Tests whether group membership and category distribution are independent, using a contingency table.
- **Bonferroni correction** (post-hoc): With three groups there are three pairwise comparisons. The significance threshold is divided by the number of pairs (α/3 ≈ 0.017) to control the family-wise error rate.

**Table title (suggestion):**
> *Demographic and clinical characteristics of the study groups. Values are Median (IQR) for continuous variables and n (%) for categorical variables. Global p-values from Kruskal-Wallis test (continuous) or chi-squared test (categorical). Post-hoc comparisons Bonferroni-corrected.*

**Footnote:**
> \* Disease duration and age at diagnosis are reported for de novo PD only; no cross-group comparison was performed.

**Group codes:** 1 = de novo PD, 2 = HC, 4 = prodromal PD